In [ ]:
import ee 
import luma_ge
luma_ge.authenticate_manually()
import geemap

# Step 1: Select avaliable imagery

In [ ]:
#retrieve aoi from assets
aoi = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi1_AOI_Kambera_NTT')
#import required library and initialize them
from luma_ge.data_acquisition import Reflectance_Data, Reflectance_Stats, final_Image
optical_reflectance = Reflectance_Data()
composite = final_Image()
#search cloud free data
start = '2025-01-01'
end = '2025-12-31'
#get the image collection and corresponding statistics
landsat_9, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L9_SR', 
                                                           cloud_cover=20, compute_detailed_stats=False)
# Compute the median composite
landsat_median = landsat_9.median().clip(aoi)
#visualization parameter
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}
#Add the data to the map
Map = geemap.Map()
Map.addLayer(landsat_median, l8_sr_visparam, 'L8 SR Image Collection')
# set center of the map in the area of interest
Map.centerObject(aoi, 12)
#Map

# Step 2: Feature Engineering
## Topography based predictor

In [ ]:
#import and initialize the required modules
from luma_ge.predictor import terrain_calculator, SpectralCalculator, distance_calculator
dist_calc = distance_calculator()
terrain_calc = terrain_calculator()
spectral_calc = SpectralCalculator()
#Define DEM
#retrieve topography based predictors
dem_source = 'NASADEM'
elevation = terrain_calc.calculate_elevation(aoi, dem_source=dem_source) # type: ignore
slope = terrain_calc.calculate_slope(aoi, dem_source=dem_source) # type: ignore
aspect = terrain_calc.calculate_aspect(aoi, dem_source=dem_source) # type: ignore
#Visualize terrain layers on the map
terrain_vis = {
    'min': 0,
    'max': 500,
    'palette': ['black', 'blue', 'green', 'yellow', 'red']}
slope_vis = {
    'min': 0,
    'max': 45,
    'palette': ['white', 'lightblue', 'green', 'yellow', 'red']}
aspect_vis = {
    'min': 0,
    'max': 360,
    'palette': ['white', 'blue', 'green', 'yellow', 'red', 'purple']
}
m = geemap.Map()
m.addLayer(elevation, terrain_vis, f"Elevation ({dem_source})")
m.addLayer(slope, slope_vis, f"Slope ({dem_source})")

## Spectral Transformation

In [ ]:
#Compute spectral indices from the optical collection
indices_to_compute = ['EVI', 'GBNDVI', 'MSAVI', 'GNDVI', 'SAVI', 'NDVI', 'MNDWI', 'NDBI', 'MBI', 'NDMI' ]
spectral_indices = spectral_calc.calculate_indices_with_collection(
    collection=landsat_9,
    aoi=aoi, # type: ignore
    index_list=indices_to_compute,
    reducer_method='mean')
#check the computed variables
print('Computed terrain layers: elevation, slope, aspect')
print('Computed spectral indices:', spectral_indices.bandNames().getInfo() if spectral_indices else 'failed')
if spectral_indices:
    ndvi_vis = {'min': -1, 'max': 1, 'palette': ['purple', 'white', 'green']}
    evi_vis = {'min': -1, 'max': 1, 'palette': ['navy', 'white', 'lime']}
    m.addLayer(spectral_indices.select('GBNDVI'), ndvi_vis, 'GBNDVI')
    m.addLayer(spectral_indices.select('EVI'), evi_vis, 'EVI')
Map.centerObject(aoi, 12)

## Distance based predictor

In [ ]:
#distance based predictors (5km buffer)
distance_predictor = dist_calc.calculate_distance_metrics(
    aoi = aoi.geometry(), max_dist = 50000, in_meters = True)

## Stack all of the predictors/features

In [ ]:
#stack the predictors and the imagery
#distance .addBands(distance_predictor) is the bottleneck for the RFE calculation
#handle by predictor improvement branch
predictor_stack = landsat_median.addBands(spectral_indices).addBands(elevation).addBands(slope).addBands(aspect).toFloat()
print('Final predictor stack bands:', predictor_stack.bandNames().getInfo() if predictor_stack else 'failed')

# Step 3: Extract the pixel value based on the training data

In [ ]:
from luma_ge.classification import FeatureExtraction, Generate_LULC
labeled_roi = ee.FeatureCollection('projects/ee-agilfahrezy60/assets/Sesi1_Data_Latih_Kambera')
features = FeatureExtraction()
train, test = features.stratified_split(roi=labeled_roi, 
                                        image=predictor_stack, 
                                        class_prop='ID_Kelas', 
                                        train_ratio=0.6, 
                                        seed=42)

# Step 4: Perform Recursive Feature Elimination using GEE

In [ ]:
#import rfegee class
from luma_ge.predictor import RFEGEE
predictor_name = predictor_stack.bandNames().getInfo()
rfe = RFEGEE(
    training_data=train,  #Training data
    class_property='ID_Kelas',  #class ID property
    features=predictor_name,  #Pass list of band names, not the image
    n_features_to_select=15,  #how many features to select (?)
    step=1,  #how many features remove per iteration
    adaptive = True, #Set to True to save EECU
    auto_select=True,  #Automatically keeps peak-accuracy subset
    n_trees=100, 
    test_data=test, 
    seed=42,
    track_accuracy=True,  #Track accuracy to see how performance changes
    verbose=True  #Print progress
)
#run the RFE algorithm
rfe.fit()
print("\n" + "="*60)
print("RFE Execution Complete!")
print("="*60)
#side note: time to process = 4m 31.6s

## Step 4.1: Evaluate the result

In [ ]:
#Get selected features
selected_features = rfe.get_selected_features()
print(f"Selected Features ({len(selected_features)}):")
for i, feature in enumerate(selected_features, 1):
    print(f"  {i}. {feature}")

#feature rankings (1 = selected, 2+ = order of elimination)
rankings = rfe.get_ranking()
print(f"\nFeature Rankings:")
for feature, rank in sorted(rankings.items(), key=lambda x: x[1]):
    print(f"  {feature}: Rank {rank}")
#support (boolean dict of selected features)
support = rfe.get_support()
print(f"\nFeature Support (Selected/Not Selected):")
for feature, selected in support.items():
    status = "✓ Selected" if selected else "✗ Eliminated"
    print(f"  {feature}: {status}")

In [ ]:
import matplotlib.pyplot as plt
#Feature Rankings
fig1 = rfe.plot_feature_rankings(figsize=(12, 8))
plt.show()
#Accuracy vs Number of Features (only work if track accuracy is enable)
if rfe.accuracy_history_:
    fig2 = rfe.plot_accuracy_curve(figsize=(10, 6))
    plt.show()
# Combined Summary
fig3 = rfe.plot_rfe_summary(figsize=(16, 6))
plt.show()

In [ ]:
results_df = rfe.get_results_dataframe()
print("Results DataFrame:")
print(results_df.to_string())

# Accuracy history (if available)
if rfe.accuracy_history_:
    accuracy_df = rfe.get_accuracy_dataframe()
    print("\nAccuracy History:")
    print(accuracy_df.to_string())

## Step 4.2: Retrieve the final predictor (optimum features) and perform classification

In [ ]:
# Build the final predictor stack from the optimum RFE features
selected_features = rfe.get_selected_features()
final_predictor_stack = predictor_stack.select(selected_features).toFloat()
print(f"Final features ({len(selected_features)}):")
print(final_predictor_stack.bandNames().getInfo())
#Re-extract training pixels for the selected features
final_training, final_test = features.stratified_split(roi=labeled_roi,
                                                class_prop = 'ID_Kelas',
                                                image=final_predictor_stack,
                                                pixel_size=30,
                                                train_ratio=0.7,
                                                seed=42)

#perform the classification using final predictor stack
lulc_generator = Generate_LULC()
final_classification = lulc_generator.hard_classification(
    training_data=final_training,
    class_property='ID_Kelas',
    image=final_predictor_stack,
    ntrees=100,
    seed=42
)

classification_vis = {
    'min': 1,
    'max': 6,
    'palette': ['419BDF', '397D49', '88B053', '7A87C6', 'E49635', 'DFC35A']
}
m.addLayer(final_classification, classification_vis, 'Luma-Ge Classification (RFE Features)')
m.centerObject(aoi, 12)
m

# Step 5: Local RFE with Cross Validation (python environment)

In [ ]:
from luma_ge.predictor import RFE_CV
predictor_name = predictor_stack.bandNames().getInfo()
rfe_local = RFE_CV(
    training_data=train,
    class_property='ID_Kelas',
    features=predictor_name,
    n_features_to_select=10,
    step=1,
    cv=5,
    n_trees=100,
    seed=42,
    verbose=True,
)
rfe_local.fit()

# Recommended feature stack
recommended_features = rfe_local.get_selected_features()
recommended_stack = rfe_local.select_bands(predictor_stack)
display(rfe_local.get_results_dataframe())
rfe_local.plot_feature_importance()